# Uplift Modeling & Causal Treatment Effects

Companion notebook for the [Uplift Modeling wiki page](https://ml-viz-ruby.vercel.app/wiki/uplift-modeling).

We implement S-Learner, T-Learner, and X-Learner uplift estimators, compute the Qini curve, and compare their AUUC on synthetic A/B data.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — Synthetic A/B experiment data

In [ ]:
n = 2000
# Feature: user engagement score (0-1)
X = rng.uniform(0, 1, n)

# True treatment effect: higher for engaged users (persuadables)
true_tau = np.where(X > 0.6, 0.15, np.where(X < 0.2, -0.05, 0.02))
baseline  = 0.1 + 0.3 * X    # baseline conversion probability

# Random treatment assignment (A/B test)
T = rng.binomial(1, 0.5, n)

# Observed outcome
p_treated   = np.clip(baseline + T * true_tau, 0, 1)
Y = rng.binomial(1, p_treated)

print(f"Control group:  n={( T==0).sum()}, conversion rate={Y[T==0].mean():.3f}")
print(f"Treatment group: n={(T==1).sum()}, conversion rate={Y[T==1].mean():.3f}")
print(f"Average treatment effect (ATE): {Y[T==1].mean() - Y[T==0].mean():.4f}")

## 2 — S-Learner

In [ ]:
def s_learner_predict(X, T, Y, X_test):
    """Linear S-Learner: fit one model on [X, T], predict tau as f(x,1)-f(x,0)."""
    Xf = np.column_stack([X, T])     # feature + treatment
    Xf_t = np.column_stack([np.ones(X.shape[0]), Xf])
    w = np.linalg.lstsq(Xf_t, Y, rcond=None)[0]
    
    Xt1 = np.column_stack([np.ones(len(X_test)), X_test, np.ones(len(X_test))])
    Xt0 = np.column_stack([np.ones(len(X_test)), X_test, np.zeros(len(X_test))])
    return (Xt1 @ w) - (Xt0 @ w)

tau_s = s_learner_predict(X, T, Y, X)
print(f"S-Learner: mean estimated tau = {tau_s.mean():.4f} (true ATE ≈ {true_tau.mean():.4f})")

## 3 — T-Learner

In [ ]:
def t_learner_predict(X, T, Y, X_test):
    """T-Learner: separate linear models for treatment and control."""
    def linreg(x, y):
        Xb = np.column_stack([np.ones(len(x)), x])
        return np.linalg.lstsq(Xb, y, rcond=None)[0]
    
    w0 = linreg(X[T==0], Y[T==0])
    w1 = linreg(X[T==1], Y[T==1])
    Xt = np.column_stack([np.ones(len(X_test)), X_test])
    return (Xt @ w1) - (Xt @ w0)

tau_t = t_learner_predict(X, T, Y, X)
print(f"T-Learner: mean estimated tau = {tau_t.mean():.4f} (true ATE ≈ {true_tau.mean():.4f})")

## 4 — Qini curve

In [ ]:
def qini_curve(Y, T, tau_hat, n_bins=20):
    order = np.argsort(-tau_hat)
    fracs, gains = [0], [0]
    for k in range(1, n_bins+1):
        idx = order[:k * len(order) // n_bins]
        nt = T[idx].sum()
        nc = (1-T[idx]).sum()
        if nt == 0 or nc == 0: continue
        gain = Y[idx][T[idx]==1].sum()/nt - Y[idx][T[idx]==0].sum()/nc
        fracs.append(k/n_bins)
        gains.append(gain)
    return np.array(fracs), np.array(gains)

fig, ax = plt.subplots(figsize=(7,4))
for name, tau_hat in [("S-Learner", tau_s), ("T-Learner", tau_t), ("Random", rng.uniform(-0.1,0.1,n))]:
    f, g = qini_curve(Y, T, tau_hat)
    ax.plot(f, g, label=name, lw=2)
ax.set_xlabel("Fraction treated"); ax.set_ylabel("Incremental gain")
ax.set_title("Qini Curves — Uplift Model Comparison")
ax.legend(); plt.tight_layout(); plt.show()

## ✏️ Your turn

In [ ]:
def compute_auuc(Y, T, tau_hat, n_bins=20):
    """Compute Area Under Uplift Curve (AUUC) using the Qini curve."""
    # TODO(you): get the Qini curve and compute the area using np.trapz
    return ...

for name, tau_hat in [("S-Learner", tau_s), ("T-Learner", tau_t)]:
    auuc = compute_auuc(Y, T, tau_hat)
    print(f"{name} AUUC: {auuc:.6f}")

<details><summary>Solution</summary>

```python
def compute_auuc(Y, T, tau_hat, n_bins=20):
    f, g = qini_curve(Y, T, tau_hat, n_bins)
    return np.trapz(g, f)
```
</details>